# INFO284 Semester assignment

In [21]:
import pandas as pd 
import re

import matplotlib.pyplot as plt

#Naive bayes
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

#Logistic regression
import numpy as np
from sklearn import linear_model
from sklearn.linear_model import LogisticRegression

#LSTM Model
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout





## Retrieving the Dataset

In [20]:
df_original = pd.read_csv("reviews.csv")
df = df_original.copy()
df.head()



  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 21.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 21.1 MB/s  0:00:00
Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]


## Data Analysis

## Data Cleaning

In [ ]:
rating_counts = df["rating"].value_counts().sort_index()

plt.figure()
rating_counts.plot(kind="bar")

plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.title("Rating Distribution (Before Cleaning)")
plt.show()


In [ ]:
#This filter removes reviews containing unusually long single words, which are often spam or noise, improving data quality for text analysis.

#Number of reviews before filtering
before_count = len(df)
df = df[~df["review_text"].str.split().apply(

     lambda words: any(len(word) > 15 for word in words)

 )]
   
# Number of reviews after filtering
after_count = len(df)
 
# Number of reviews removed in filtering
removed_count = before_count - after_count
 
print("Before filtering:", before_count)
print("After removing excessive characters:", after_count)
print("Removed:", removed_count)


Before filtering: 6210
After removing excessive characters: 5713
Removed: 497


In [ ]:
# Filter for removing repeating words

# Count before filtering
before_count2 = len(df)

# Regex pattern for 3+ repeated consecutive words
pattern2 = r'\b(\w+)\b(?=\s+\1\b\s+\1\b)'

# Identify rows to remove
mask = df["review_text"].str.contains(pattern2, case=False, regex=True, na=False)

removed_count2 = mask.sum()

# Filter dataset
df_clean = df[~mask]

after_count2 = len(df_clean)

# Print results
print(f"Total reviews before filtering: {before_count2}")
print(f"Reviews removed: {removed_count2}")
print(f"Total reviews after filtering: {after_count2}")



Total reviews before filtering: 5713
Reviews removed: 185
Total reviews after filtering: 5528


/var/folders/z_/gmcwgltj5cj57p3fyycqjr800000gn/T/ipykernel_6960/2246965277.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = df["review_text"].str.contains(pattern2, case=False, regex=True, na=False)


## Feature filtering

In [ ]:
# Remove irrelevant features not needed for the model
df = df.drop(columns=["review_id", "review_date", "helpful"])
df.head()

,rating,review_text
0,5,It's good
1,5,WhatsApp not working well always shows offline...
2,5,"Oppo not corresponding, share with me the offi..."
3,5,"Excellent app, great communication super conne..."
4,4,simply the ɓest for chat and calls.i love it


## Machine Learning models

### Model 1: Multinomial Naive Bayes

In [ ]:
X = df["review_text"]
y = df["rating"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer(
    ngram_range=(1,2),
    max_features=5000
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

y_pred = nb_model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


# https://www.geeksforgeeks.org/machine-learning/multinomial-naive-bayes/ kilde inspo 

Accuracy: 0.621172353455818
              precision    recall  f1-score   support

           1       0.54      0.61      0.58       287
           2       0.00      0.00      0.00        79
           3       0.14      0.01      0.02        77
           4       0.14      0.02      0.03       110
           5       0.67      0.90      0.77       590

    accuracy                           0.62      1143
   macro avg       0.30      0.31      0.28      1143
weighted avg       0.50      0.62      0.54      1143



### Model 2: Logistic Regression 

In [ ]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_vec, y_train)

y_pred_log = log_model.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

# https://www.w3schools.com/python/python_ml_logistic_regression.asp
# Reshaped for Logistic function.

Accuracy: 0.6307961504811899
              precision    recall  f1-score   support

           1       0.55      0.67      0.60       287
           2       0.00      0.00      0.00        79
           3       0.00      0.00      0.00        77
           4       0.00      0.00      0.00       110
           5       0.67      0.90      0.76       590

    accuracy                           0.63      1143
   macro avg       0.24      0.31      0.27      1143
weighted avg       0.48      0.63      0.55      1143



/Applications/anaconda3/envs/info180/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Applications/anaconda3/envs/info180/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Applications/anaconda3/envs/info180/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me

In [ ]:
vocab_size = 20000     
max_len = 200           
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding="post", truncating="post")

classes = np.sort(y_train.unique())
num_classes = len(classes)


class_to_index = {c: i for i, c in enumerate(classes)}
y_train_idx = y_train.map(class_to_index).to_numpy()
y_test_idx = y_test.map(class_to_index).to_numpy()

embedding_dim = 128

lstm_model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax")
])

lstm_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

print("\n--- LSTM model summary ---")
lstm_model.summary()



early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train_pad, y_train_idx,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


test_loss, test_acc = lstm_model.evaluate(X_test_pad, y_test_idx, verbose=0)
print("\n--- LSTM Text Classifier ---")
print("Accuracy:", test_acc)


y_pred_probs = lstm_model.predict(X_test_pad, verbose=0)
y_pred_idx = np.argmax(y_pred_probs, axis=1)


index_to_class = {i: c for c, i in class_to_index.items()}
y_pred_lstm = np.array([index_to_class[i] for i in y_pred_idx])

print(classification_report(y_test, y_pred_lstm))

# https://www.geeksforgeeks.org/deep-learning/long-short-term-memory-lstm-rnn-in-tensorflow/


--- LSTM model summary ---


/Applications/anaconda3/envs/info180/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 6s 82ms/step - accuracy: 0.4770 - loss: 1.3496 - val_accuracy: 0.5088 - val_loss: 1.3130
Epoch 2/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 5s 78ms/step - accuracy: 0.4970 - loss: 1.3143 - val_accuracy: 0.5088 - val_loss: 1.3139
Epoch 3/10
58/58 ━━━━━━━━━━━━━━━━━━━━ 5s 85ms/step - accuracy: 0.5038 - loss: 1.3049 - val_accuracy: 0.5088 - val_loss: 1.3154

--- LSTM Text Classifier ---
Accuracy: 0.516185462474823
              precision    recall  f1-score   support

           1       0.00      0.00      0.00       287
           2       0.00      0.00      0.00        79
           3       0.00      0.00      0.00        77
           4       0.00      0.00      0.00       110
           5       0.52      1.00      0.68       590

    accuracy                           0.52      1143
   macro avg       0.10      0.20      0.14      1143
weighted avg       0.27      0.52      0.35      1143



/Applications/anaconda3/envs/info180/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Applications/anaconda3/envs/info180/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Applications/anaconda3/envs/info180/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me